# Melhores Compras — Atividade 1: reconhecimento facial

**Objetivo:** identificar cinco pessoas usando quatro fotografias de referência por pessoa e uma fotografia inédita para teste.

**Método:** o modelo pré-treinado FaceNet512 extrai um vetor numérico (*embedding*) de cada rosto. Depois, comparamos os vetores usando **similaridade de cossenos**. A identidade prevista é aquela com **maior similaridade média** entre as quatro fotografias de referência.

> Não há treinamento de uma rede neural do zero: as quatro imagens de cada pessoa formam a base de referência, conforme a abordagem de comparação permitida no enunciado.

## 1. Preparar as fotografias

Coloque **5 pessoas**, com **4 fotos em `treino` e 1 foto em `teste`**. Pode renomear `pessoa_01`, `pessoa_02` etc. para os nomes desejados. O código suporta os formatos: `.jpg`, `.jpeg` e `.png`.

```text
fotos/
├── pessoa_01/
├── pessoa_02/
├── pessoa_03/
├── pessoa_04/
└── pessoa_05/
│   ├── treino/   ← 4 fotografias
│   └── teste/    ← 1 fotografia diferente
```

## 2. Imports e configurações

In [ ]:
%pip install deepface numpy pandas matplotlib Pillow jupyterlab ipykernel

In [1]:
from pathlib import Path
import os

# Usa o backend leve ONNX nas versões atuais do DeepFace.
os.environ["DEEPFACE_BACKEND_ENGINE"] = "onnx"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
from deepface import DeepFace

PASTA_FOTOS = Path("fotos")
MODELO = "Facenet512"
DETECTOR = "opencv"
EXTENSOES = {".jpg", ".jpeg", ".png"}

In [ ]:
from pathlib import Path

print(Path.cwd())

## 3. Conferir a base de imagens

In [ ]:
if not PASTA_FOTOS.exists():
    raise FileNotFoundError("Pasta 'fotos' não encontrada.")

pessoas = sorted(p for p in PASTA_FOTOS.iterdir() if p.is_dir())
if len(pessoas) != 5:
    raise ValueError(f"Precisamos de exatamente 5 pessoas, mas temos: {len(pessoas)} pessoas.")

base = {}
for pasta in pessoas:
    treino = sorted(f for f in (pasta / "treino").glob("*") if f.suffix.lower() in EXTENSOES)
    teste = sorted(f for f in (pasta / "teste").glob("*") if f.suffix.lower() in EXTENSOES)

    if len(treino) != 4 or len(teste) != 1:
        raise ValueError(f"{pasta.name}: esperado 4 fotos de treino e 1 de teste. "
                         f"Encontradas: {len(treino)} treino, {len(teste)} teste.")
    base[pasta.name] = {"treino": treino, "teste": teste[0]}

conferencia = pd.DataFrame([
    {"Pessoa": nome, "Fotos de referência": len(fotos["treino"]), "Fotos de teste": 1}
    for nome, fotos in base.items()
])
display(conferencia)
print("Base conferida: 5 pessoas, 20 fotos de referência e 5 de teste.")

## 4. Visualizar as fotografias

In [ ]:
fig, eixos = plt.subplots(5, 5, figsize=(13, 13))

for linha, (nome, fotos) in enumerate(base.items()):
    imagens = fotos["treino"] + [fotos["teste"]]
    for coluna, caminho in enumerate(imagens):
        with Image.open(caminho) as imagem:
            eixos[linha, coluna].imshow(imagem.convert("RGB"))
        eixos[linha, coluna].set_title(f"{nome} | {'teste' if coluna == 4 else 'treino'}", fontsize=9)
        eixos[linha, coluna].axis("off")

plt.tight_layout()
plt.show()

## 5. Detectar rostos e extrair embeddings

O DeepFace detecta e alinha os rostos antes de extrair os embeddings com FaceNet512. Utilizamos enforce_detection=True e, quando há múltiplas detecções, selecionamos a região facial de menor área. Essa escolha é uma heurística específica do experimento, não uma regra geral para reconhecimento facial.

In [ ]:
def extrair_embedding(caminho):
    rostos = DeepFace.represent(
        img_path=str(caminho),
        model_name=MODELO,
        detector_backend=DETECTOR,
        enforce_detection=True,
        align=True,
    )

    if len(rostos) > 1:
        print(f"Aviso: {len(rostos)} rostos detectados em {caminho.name}")

    rosto = min(
        rostos,
        key=lambda r: r["facial_area"]["w"] * r["facial_area"]["h"]
    )

    return np.array(rosto["embedding"], dtype=float)

referencias = {}
for nome, fotos in base.items():
    referencias[nome] = []
    for caminho in fotos["treino"]:
        referencias[nome].append(extrair_embedding(caminho))
    print(f"{nome}: 4 rostos processados.")

print("\nTodas as 20 imagens de referência foram processadas.")

## 6. Comparar a fotografia de teste usando cossenos

A similaridade de cossenos vai de **−1 a 1**, e valores maiores indicam vetores mais próximos. Para cada pessoa, calculamos a **média das quatro similaridades** e escolhemos a maior. Isso é uma classificação entre as cinco identidades conhecidas, **não uma autenticação de segurança**.

In [ ]:
def similaridade_cosseno(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

resultados = []
placares = []

for nome_correto, fotos in base.items():
    teste = extrair_embedding(fotos["teste"])

    medias = {}
    for nome, vetores in referencias.items():
        notas = [similaridade_cosseno(teste, vetor) for vetor in vetores]
        medias[nome] = float(np.mean(notas))

    previsao = max(medias, key=medias.get)
    resultados.append({
        "Pessoa real": nome_correto,
        "Pessoa prevista": previsao,
        "Similaridade média": round(medias[previsao], 4),
        "Acertou": nome_correto == previsao,
    })
    placares.append(medias)

resumo = pd.DataFrame(resultados)
matriz = pd.DataFrame(placares, index=list(base))

display(resumo)
acertos = int(resumo["Acertou"].sum())
print(f"Resultado: {acertos}/5 pessoas reconhecidas corretamente ({acertos / 5:.0%} de acerto).")

## 7. Visualizar a matriz de similaridade

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
mapa = ax.imshow(matriz.values, cmap="viridis", vmin=-1, vmax=1)
ax.set_xticks(range(5), labels=matriz.columns, rotation=30, ha="right")
ax.set_yticks(range(5), labels=matriz.index)
ax.set_xlabel("Pessoa na base de referência")
ax.set_ylabel("Pessoa na foto de teste")
ax.set_title("Similaridade média de cossenos")

for i in range(5):
    for j in range(5):
        ax.text(j, i, f"{matriz.iloc[i, j]:.2f}", ha="center", va="center", color="white")

fig.colorbar(mapa, ax=ax, label="Similaridade (−1 a 1)")
plt.tight_layout()

PASTA_RESULTADOS = Path("resultados")
PASTA_RESULTADOS.mkdir(exist_ok=True)
fig.savefig(PASTA_RESULTADOS / "matriz_similaridade.png", dpi=160, bbox_inches="tight")
plt.show()

resumo.to_csv(PASTA_RESULTADOS / "resultados_reconhecimento.csv", index=False, encoding="utf-8-sig")
matriz.to_csv(PASTA_RESULTADOS / "similaridades_por_pessoa.csv", encoding="utf-8-sig")
print("Resultados salvos na pasta 'resultados'.")

## 8. Conclusão

In [ ]:
print(f"""Algoritmo: FaceNet512 pré-treinado + similaridade de cossenos.
Método: 4 fotografias de referência por pessoa; classificação da 5ª foto
pela maior média de similaridade com as 4 referências de cada identidade.
Resultado: {acertos} acertos em 5 testes ({acertos / 5:.0%}).
Avaliação: {'todos os testes foram identificados corretamente' if acertos == 5 else 'houve erros de identificação que precisam ser discutidos'}.
Limitações: amostra pequena, condições das fotos e ausência de validação contra
fraudes com fotografia/vídeo. O experimento não comprova segurança para uso real.
""")